# DeepPoly: Certifying Neural Network Robustness on MNIST

This notebook trains a small fully-connected network on MNIST, attacks it with FGSM, then formally certifies per-image robustness using the DeepPoly abstract domain.

**Reference:** Singh et al., *An Abstract Domain for Certifying Neural Networks*, POPL 2019.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import struct, gzip, urllib.request, os, time
from pathlib import Path

## 1. Load MNIST

In [ ]:
BASE = "https://ossci-datasets.s3.amazonaws.com/mnist"
FILES = [
    "train-images-idx3-ubyte.gz",
    "train-labels-idx1-ubyte.gz",
    "t10k-images-idx3-ubyte.gz",
    "t10k-labels-idx1-ubyte.gz",
]

os.makedirs("mnist", exist_ok=True)
for fname in FILES:
    out = Path("mnist") / fname[:-3]
    if not out.exists():
        print(f"Downloading {fname}...")
        urllib.request.urlretrieve(f"{BASE}/{fname}", str(out) + ".gz")
        with gzip.open(str(out) + ".gz", "rb") as f_in, open(out, "wb") as f_out:
            f_out.write(f_in.read())
        os.remove(str(out) + ".gz")
    else:
        print(f"Already have {out.name}")

def read_idx_images(path):
    with open(path, "rb") as f:
        magic, n, r, c = struct.unpack(">IIII", f.read(16))
        data = np.frombuffer(f.read(), dtype=np.uint8)
    return data.reshape(n, r * c).astype(np.float32) / 255.0

def read_idx_labels(path):
    with open(path, "rb") as f:
        magic, n = struct.unpack(">II", f.read(8))
        data = np.frombuffer(f.read(), dtype=np.uint8)
    return data.astype(np.int32)

X_train = read_idx_images("mnist/train-images-idx3-ubyte")
y_train = read_idx_labels("mnist/train-labels-idx1-ubyte")
X_test  = read_idx_images("mnist/t10k-images-idx3-ubyte")
y_test  = read_idx_labels("mnist/t10k-labels-idx1-ubyte")

print(f"Train: {X_train.shape}  Test: {X_test.shape}")

## 2. Network — 784 → 128 → 64 → 10

Hidden layers use ReLU. Output uses sigmoid (so outputs are in (0,1), suitable for MSE against one-hot targets).

In [ ]:
def sigmoid(x):      return 1.0 / (1.0 + np.exp(-np.clip(x, -500, 500)))
def sigmoid_d(s):    return s * (1.0 - s)   # s = sigmoid(pre-activation)
def relu(x):         return np.maximum(0.0, x)
def relu_d(x):       return (x > 0).astype(np.float32)

class Net:
    def __init__(self, dims, scale=0.1):
        rng = np.random.default_rng(42)
        self.W = [rng.uniform(-scale, scale, (dims[i], dims[i+1])).astype(np.float32)
                  for i in range(len(dims)-1)]
        self.b = [np.zeros((1, dims[i+1]), dtype=np.float32)
                  for i in range(len(dims)-1)]

    def forward(self, X):
        self.zs, self.as_ = [], [X]
        a = X
        for i, (W, b) in enumerate(zip(self.W, self.b)):
            z = a @ W + b
            self.zs.append(z)
            a = sigmoid(z) if i == len(self.W)-1 else relu(z)
            self.as_.append(a)
        return a

    def predict(self, X):
        return np.argmax(self.forward(X), axis=1)

    def loss(self, X, y_onehot):
        out = self.forward(X)
        return np.mean((out - y_onehot) ** 2)

    def backward(self, y_onehot, lr):
        n = self.as_[0].shape[0]
        # Output layer delta (MSE + sigmoid)
        delta = 2.0 * (self.as_[-1] - y_onehot) / n * sigmoid_d(self.as_[-1])
        grads_W, grads_b = [], []
        for i in reversed(range(len(self.W))):
            grads_W.insert(0, self.as_[i].T @ delta)
            grads_b.insert(0, delta.sum(axis=0, keepdims=True))
            if i > 0:
                delta = (delta @ self.W[i].T) * relu_d(self.zs[i-1])
            else:
                self._input_grad = delta @ self.W[0].T  # for FGSM
        for i in range(len(self.W)):
            self.W[i] -= lr * grads_W[i]
            self.b[i]  -= lr * grads_b[i]

net = Net([784, 128, 64, 10])
print("Network ready: 784 → 128 → 64 → 10  (ReLU hidden, Sigmoid output)")

## 3. Train (SGD, mini-batch, 30 epochs)

In [ ]:
EPOCHS   = 30
LR       = 0.5
BATCH    = 32
rng      = np.random.default_rng(0)

def onehot(y, n=10):
    oh = np.zeros((len(y), n), dtype=np.float32)
    oh[np.arange(len(y)), y] = 1.0
    return oh

history = []
for epoch in range(EPOCHS):
    t0  = time.time()
    idx = rng.permutation(len(X_train))
    losses = []
    for start in range(0, len(X_train), BATCH):
        batch = idx[start:start+BATCH]
        Xb    = X_train[batch]
        yb    = onehot(y_train[batch])
        net.forward(Xb)
        losses.append(net.loss(Xb, yb))
        net.backward(yb, LR)
    acc = np.mean(net.predict(X_test) == y_test) * 100
    history.append((np.mean(losses), acc))
    print(f"Epoch {epoch+1:2d}/{EPOCHS} | Loss: {np.mean(losses):.5f} | "
          f"Test Acc: {acc:.2f}% | {time.time()-t0:.1f}s")

print(f"\nFinal test accuracy: {history[-1][1]:.2f}%")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.5))
epochs_x = range(1, EPOCHS+1)
ax1.plot(epochs_x, [h[0] for h in history], color='#e06c6c', lw=2)
ax1.set(title='Training Loss', xlabel='Epoch', ylabel='MSE')
ax1.grid(alpha=0.3)
ax2.plot(epochs_x, [h[1] for h in history], color='#4fa3e0', lw=2)
ax2.set(title='Test Accuracy', xlabel='Epoch', ylabel='Accuracy (%)')
ax2.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 4. FGSM Attack

**Fast Gradient Sign Method** constructs an adversarial example by perturbing each pixel in the direction of the loss gradient:

$$x_{adv} = \text{clip}\bigl(x + \varepsilon \cdot \text{sign}(\nabla_x L),\; 0,\; 1\bigr)$$

The perturbation is bounded by $\varepsilon$ in $L_\infty$ norm, so it's imperceptible but can flip the prediction.

In [ ]:
def fgsm_attack(net, x, y_true, epsilon):
    oh = onehot([y_true])
    net.forward(x.reshape(1, -1))
    net.backward(oh, lr=0)       # lr=0: compute grads, don't update weights
    grad = net._input_grad.ravel()
    return np.clip(x + epsilon * np.sign(grad), 0.0, 1.0)

N_IMAGES  = 100
EPS_ATK   = 0.10   # attack strength

results = []
for i in range(N_IMAGES):
    x, y = X_test[i], int(y_test[i])
    pred_clean = int(net.predict(x.reshape(1,-1))[0])
    x_adv      = fgsm_attack(net, x, y, EPS_ATK)
    pred_adv   = int(net.predict(x_adv.reshape(1,-1))[0])
    attacked   = (pred_clean == y) and (pred_adv != y)
    results.append(dict(idx=i, true=y, pred_clean=pred_clean,
                        pred_adv=pred_adv, attacked=attacked,
                        x=x, x_adv=x_adv, robust=False, margin=0.0))

n_correct  = sum(r['pred_clean'] == r['true'] for r in results)
n_attacked = sum(r['attacked'] for r in results)
print(f"{N_IMAGES} images | {n_correct} correctly classified | {n_attacked} successfully attacked")

## 5. DeepPoly Verifier

Instead of checking every possible perturbation (exponentially many), DeepPoly propagates a **symbolic abstract element** through the network.

Each neuron $i$ maintains:
- **Symbolic bounds** expressed as linear functions over the input pixels: $\hat{l}_i(x) \le x_i \le \hat{u}_i(x)$
- **Concrete interval** $[l_i, u_i]$ obtained by evaluating the symbolic bounds over the $L_\infty$ input ball

**Three transformers:**

| Layer | How |
|---|---|
| Affine | Exact: substitute symbolic bounds of previous layer |
| ReLU | $u\le0$: zero; $l\ge0$: identity; mixed: convex hull approximation |
| Sigmoid | Monotone: apply $\sigma$ directly to concrete bounds |

**Certified robust** iff $\hat{l}_{\text{true}} > \hat{u}_j$ for every competing class $j$.

In [ ]:
def eval_concrete(lb_coef, ub_coef, lb_bias, ub_bias, inp_l, inp_u):
    """Evaluate concrete bounds from symbolic expressions over the input box."""
    l = lb_bias + np.where(lb_coef >= 0, lb_coef * inp_l, lb_coef * inp_u).sum(axis=1)
    u = ub_bias + np.where(ub_coef >= 0, ub_coef * inp_u, ub_coef * inp_l).sum(axis=1)
    return l, u

def dp_first_affine(W, b, inp_l, inp_u):
    """First layer: input neurons have identity symbolic bounds."""
    # W: (n_inp, n_out), b: (n_out,)
    lb_coef = W.T.copy()   # (n_out, n_inp)
    ub_coef = W.T.copy()
    lb_bias = b.copy()
    ub_bias = b.copy()
    l, u = eval_concrete(lb_coef, ub_coef, lb_bias, ub_bias, inp_l, inp_u)
    return lb_coef, ub_coef, lb_bias, ub_bias, l, u

def dp_affine(prev, W, b, inp_l, inp_u):
    """General affine transformer via backsubstitution."""
    p_lbc, p_ubc, p_lbb, p_ubb = prev[:4]
    # W: (n_prev, n_out)
    W_pos = np.maximum(W, 0)   # (n_prev, n_out)
    W_neg = np.minimum(W, 0)
    lb_coef = (W_pos.T @ p_lbc) + (W_neg.T @ p_ubc)  # (n_out, n_inp)
    ub_coef = (W_pos.T @ p_ubc) + (W_neg.T @ p_lbc)
    lb_bias = (W_pos.T @ p_lbb) + (W_neg.T @ p_ubb) + b
    ub_bias = (W_pos.T @ p_ubb) + (W_neg.T @ p_lbb) + b
    l, u = eval_concrete(lb_coef, ub_coef, lb_bias, ub_bias, inp_l, inp_u)
    return lb_coef, ub_coef, lb_bias, ub_bias, l, u

def dp_relu(prev, inp_l, inp_u):
    """ReLU transformer: three cases from the paper."""
    p_lbc, p_ubc, p_lbb, p_ubb, pl, pu = prev
    n_out, n_inp = p_lbc.shape
    lb_coef = np.zeros_like(p_lbc)
    ub_coef = np.zeros_like(p_ubc)
    lb_bias = np.zeros(n_out, dtype=np.float32)
    ub_bias = np.zeros(n_out, dtype=np.float32)
    l_out   = np.zeros(n_out, dtype=np.float32)
    u_out   = np.zeros(n_out, dtype=np.float32)

    for i in range(n_out):
        li, ui = pl[i], pu[i]
        if ui <= 0:
            pass  # zero: all zeros
        elif li >= 0:
            # identity (exact)
            lb_coef[i] = p_lbc[i]; ub_coef[i] = p_ubc[i]
            lb_bias[i] = p_lbb[i]; ub_bias[i] = p_ubb[i]
            l_out[i] = li; u_out[i] = ui
        else:
            # mixed: convex hull approximation
            slope     = ui / (ui - li)
            intercept = -ui * li / (ui - li)
            ub_coef[i] = slope * p_ubc[i]
            ub_bias[i] = slope * p_ubb[i] + intercept
            u_out[i]   = ui
            if ui <= -li:        # lower bound = 0 (smaller area)
                pass
            else:                # lower bound = identity (smaller area)
                lb_coef[i] = p_lbc[i]; lb_bias[i] = p_lbb[i]; l_out[i] = li

    return lb_coef, ub_coef, lb_bias, ub_bias, l_out, u_out

def deeppoly_verify(net, x, true_label, epsilon):
    inp_l = np.clip(x - epsilon, 0.0, 1.0).astype(np.float32)
    inp_u = np.clip(x + epsilon, 0.0, 1.0).astype(np.float32)

    n_layers = len(net.W)

    # First layer
    W0 = net.W[0]; b0 = net.b[0].ravel()
    curr = dp_first_affine(W0, b0, inp_l, inp_u)
    if n_layers > 1:
        curr = dp_relu(curr, inp_l, inp_u)
    else:
        *rest, l, u = curr
        curr = (*rest, sigmoid(l), sigmoid(u))

    # Remaining layers
    for layer in range(1, n_layers):
        is_out = (layer == n_layers - 1)
        W = net.W[layer]; b = net.b[layer].ravel()
        curr = dp_affine(curr, W, b, inp_l, inp_u)
        if not is_out:
            curr = dp_relu(curr, inp_l, inp_u)
        else:
            *rest, l, u = curr
            curr = (*rest, sigmoid(l), sigmoid(u))

    _, _, _, _, l_out, u_out = curr
    margin = min(l_out[true_label] - u_out[j]
                 for j in range(10) if j != true_label)
    return margin > 0, float(margin)

print("DeepPoly verifier ready.")

## 6. Run Verification

In [ ]:
EPS_VERIFY = 0.05

for r in results:
    robust, margin = deeppoly_verify(net, r['x'], r['true'], EPS_VERIFY)
    r['robust'] = robust
    r['margin'] = margin

n_robust = sum(r['robust'] for r in results)
print(f"{N_IMAGES} images | {n_correct} correct | {n_attacked} attacked (ε={EPS_ATK}) | {n_robust} certified robust (ε={EPS_VERIFY})")

## 7. Results Grid

In [ ]:
COLS = 10
ROWS = 10
fig, axes = plt.subplots(ROWS, COLS, figsize=(16, 16))
fig.patch.set_facecolor('#1a1a2e')

COLOR = {'robust': '#32dc50', 'attacked': '#dc3232', 'unverif': '#c8b400', 'wrong': '#888888'}

for i, r in enumerate(results[:ROWS*COLS]):
    ax = axes[i // COLS][i % COLS]
    if r['attacked']:
        img   = r['x_adv'].reshape(28, 28)
        color = COLOR['attacked']
        label = f"[{r['true']}]→{r['pred_adv']}"
        status = 'ATTACKED'
    elif r['pred_clean'] != r['true']:
        img   = r['x'].reshape(28, 28)
        color = COLOR['wrong']
        label = f"[{r['true']}]→{r['pred_clean']}"
        status = 'WRONG'
    elif r['robust']:
        img   = r['x'].reshape(28, 28)
        color = COLOR['robust']
        label = f"[{r['true']}]→{r['pred_clean']}"
        status = 'ROBUST'
    else:
        img   = r['x'].reshape(28, 28)
        color = COLOR['unverif']
        label = f"[{r['true']}]→{r['pred_clean']}"
        status = 'UNVERIF'

    ax.imshow(img, cmap='gray', vmin=0, vmax=1, interpolation='nearest')
    ax.set_facecolor('#1a1a2e')
    for spine in ax.spines.values():
        spine.set_edgecolor(color)
        spine.set_linewidth(2.5)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f"{label}\n{status}", fontsize=6.5, color=color,
                 fontfamily='monospace', pad=2)

patches = [
    mpatches.Patch(color=COLOR['robust'],   label=f'Certified Robust ({n_robust})')  ,
    mpatches.Patch(color=COLOR['attacked'], label=f'Attacked ({n_attacked})')        ,
    mpatches.Patch(color=COLOR['unverif'],  label='Unverified')                      ,
    mpatches.Patch(color=COLOR['wrong'],    label='Misclassified')                   ,
]
fig.legend(handles=patches, loc='lower center', ncol=4,
           facecolor='#1a1a2e', labelcolor='white',
           fontsize=9, framealpha=0.9, bbox_to_anchor=(0.5, -0.01))
fig.suptitle(
    f"DeepPoly on MNIST   |   attack ε={EPS_ATK}   verify ε={EPS_VERIFY}   |   "
    f"{n_correct}/{N_IMAGES} correct   {n_attacked} attacked   {n_robust} certified",
    color='white', fontsize=11, y=1.01
)
plt.tight_layout()
plt.show()

## 8. Visualise the Perturbation

In [ ]:
attacked_examples = [r for r in results if r['attacked']][:8]

fig, axes = plt.subplots(3, len(attacked_examples), figsize=(15, 5))
fig.patch.set_facecolor('#1a1a2e')
row_labels = ['Original', 'Adversarial', 'Perturbation (×5)']

for col, r in enumerate(attacked_examples):
    orig  = r['x'].reshape(28, 28)
    adv   = r['x_adv'].reshape(28, 28)
    delta = np.clip((adv - orig) * 5 + 0.5, 0, 1)

    for row, (img, cmap) in enumerate([(orig, 'gray'), (adv, 'gray'), (delta, 'RdBu_r')]):
        ax = axes[row][col]
        ax.imshow(img, cmap=cmap, vmin=0, vmax=1, interpolation='nearest')
        ax.set_facecolor('#1a1a2e')
        ax.set_xticks([]); ax.set_yticks([])
        if col == 0:
            ax.set_ylabel(row_labels[row], color='white', fontsize=9)
        if row == 0:
            ax.set_title(f"true={r['true']} → pred={r['pred_adv']}",
                         color='#e06c6c', fontsize=8.5)

fig.suptitle(f"FGSM Adversarial Examples (ε={EPS_ATK})", color='white', fontsize=12)
plt.tight_layout()
plt.show()

## 9. Certification Summary

In [ ]:
robust_margins  = [r['margin'] for r in results if r['robust']]
unverif_margins = [r['margin'] for r in results
                   if not r['robust'] and r['pred_clean'] == r['true'] and not r['attacked']]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Pie
labels  = ['Certified Robust', 'Attacked', 'Unverified', 'Misclassified']
sizes   = [
    sum(r['robust'] for r in results),
    sum(r['attacked'] for r in results),
    sum(not r['robust'] and r['pred_clean']==r['true'] and not r['attacked'] for r in results),
    sum(r['pred_clean'] != r['true'] for r in results),
]
colors  = ['#32dc50', '#dc3232', '#c8b400', '#888888']
axes[0].pie(sizes, labels=labels, colors=colors, autopct='%1.0f%%',
            textprops={'color': 'white'}, startangle=90)
axes[0].set_facecolor('#1a1a2e')
axes[0].set_title(f'N={N_IMAGES} images', color='white')
fig.patch.set_facecolor('#1a1a2e')

# Margin histogram
if robust_margins:
    axes[1].hist(robust_margins, bins=15, color='#32dc50', alpha=0.8, label='Certified')
if unverif_margins:
    axes[1].hist(unverif_margins, bins=15, color='#c8b400', alpha=0.7, label='Unverified')
axes[1].axvline(0, color='white', lw=1, ls='--')
axes[1].set_facecolor('#1a1a2e')
axes[1].tick_params(colors='white')
axes[1].set_xlabel('Certification Margin', color='white')
axes[1].set_ylabel('Count', color='white')
axes[1].set_title('Margin Distribution  (> 0 = certified)', color='white')
axes[1].legend(facecolor='#1a1a2e', labelcolor='white')
axes[1].spines[:].set_color('#555')

plt.tight_layout()
plt.show()

print(f"\nSummary  (N={N_IMAGES})")
print(f"  Correct predictions  : {n_correct}")
print(f"  Attack success (ε={EPS_ATK}) : {n_attacked}")
print(f"  Certified robust (ε={EPS_VERIFY}): {n_robust}")
if robust_margins:
    print(f"  Avg margin (robust) : {np.mean(robust_margins):.4f}")